# Dataset Analysis Notebook

This notebook provides a comprehensive framework for analyzing datasets.

## Table of Contents
1. Setup and Imports
2. Data Loading
3. Initial Data Exploration
4. Data Cleaning
5. Exploratory Data Analysis (EDA)
6. Statistical Analysis
7. Data Visualization
8. Conclusions

## 1. Setup and Imports

In [ ]:
# Data manipulation and analysis
import pandas as pd
import numpy as np

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Statistical analysis
from scipy import stats

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

print('Libraries imported successfully!')

## 2. Data Loading

Load your dataset from various formats (CSV, Excel, JSON, etc.)

In [ ]:
# Load data - Update the file path and format as needed

# For CSV files:
# df = pd.read_csv('your_dataset.csv')

# For Excel files:
# df = pd.read_excel('your_dataset.xlsx', sheet_name='Sheet1')

# For JSON files:
# df = pd.read_json('your_dataset.json')

# For demonstration, create a sample dataset
np.random.seed(42)
df = pd.DataFrame({
    'id': range(1, 101),
    'category': np.random.choice(['A', 'B', 'C', 'D'], 100),
    'value': np.random.normal(100, 15, 100),
    'quantity': np.random.randint(1, 50, 100),
    'date': pd.date_range('2024-01-01', periods=100, freq='D'),
    'score': np.random.uniform(0, 100, 100)
})

print(f'Dataset loaded successfully! Shape: {df.shape}')

## 3. Initial Data Exploration

In [ ]:
# Display first few rows
print('First 5 rows:')
display(df.head())

In [ ]:
# Display last few rows
print('Last 5 rows:')
display(df.tail())

In [ ]:
# Dataset information
print('Dataset Information:')
print(f'Number of rows: {df.shape[0]}')
print(f'Number of columns: {df.shape[1]}')
print('\nColumn details:')
df.info()

In [ ]:
# Data types
print('Data Types:')
display(df.dtypes)

In [ ]:
# Basic statistics for numerical columns
print('Statistical Summary:')
display(df.describe())

In [ ]:
# Check for missing values
print('Missing Values:')
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Percentage': missing_pct
})
display(missing_df[missing_df['Missing Count'] > 0])

if missing.sum() == 0:
    print('No missing values found!')

In [ ]:
# Check for duplicate rows
duplicates = df.duplicated().sum()
print(f'Number of duplicate rows: {duplicates}')
if duplicates > 0:
    print('\nDuplicate rows:')
    display(df[df.duplicated(keep=False)])

## 4. Data Cleaning

In [ ]:
# Create a copy for cleaning
df_clean = df.copy()

# Handle missing values (customize based on your needs)
# df_clean = df_clean.dropna()  # Drop rows with missing values
# df_clean['column_name'].fillna(df_clean['column_name'].mean(), inplace=True)  # Fill with mean
# df_clean['column_name'].fillna(df_clean['column_name'].median(), inplace=True)  # Fill with median
# df_clean['column_name'].fillna('Unknown', inplace=True)  # Fill with custom value

# Remove duplicates
df_clean = df_clean.drop_duplicates()

print(f'Cleaned dataset shape: {df_clean.shape}')
print(f'Rows removed: {df.shape[0] - df_clean.shape[0]}')

## 5. Exploratory Data Analysis (EDA)

In [ ]:
# Analyze categorical columns
categorical_cols = df_clean.select_dtypes(include=['object', 'category']).columns

print('Categorical Columns Analysis:')
for col in categorical_cols:
    print(f'\n{col}:')
    print(f'Unique values: {df_clean[col].nunique()}')
    print('Value counts:')
    display(df_clean[col].value_counts())

In [ ]:
# Analyze numerical columns
numerical_cols = df_clean.select_dtypes(include=[np.number]).columns

print('Numerical Columns Analysis:')
for col in numerical_cols:
    print(f'\n{col}:')
    print(f'Mean: {df_clean[col].mean():.2f}')
    print(f'Median: {df_clean[col].median():.2f}')
    print(f'Std Dev: {df_clean[col].std():.2f}')
    print(f'Min: {df_clean[col].min():.2f}')
    print(f'Max: {df_clean[col].max():.2f}')

In [ ]:
# Correlation matrix for numerical columns
if len(numerical_cols) > 1:
    print('Correlation Matrix:')
    correlation_matrix = df_clean[numerical_cols].corr()
    display(correlation_matrix)

## 6. Statistical Analysis

In [ ]:
# Detect outliers using IQR method
def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    return outliers, lower_bound, upper_bound

print('Outlier Detection (IQR method):')
for col in numerical_cols:
    outliers, lower, upper = detect_outliers_iqr(df_clean, col)
    print(f'\n{col}:')
    print(f'  Outliers found: {len(outliers)}')
    print(f'  Valid range: [{lower:.2f}, {upper:.2f}]')
    if len(outliers) > 0 and len(outliers) <= 10:
        print(f'  Outlier values: {outliers[col].tolist()}')

In [ ]:
# Normality tests (Shapiro-Wilk test)
print('Normality Tests (Shapiro-Wilk):')
for col in numerical_cols:
    if len(df_clean[col].dropna()) >= 3:  # Minimum sample size
        statistic, p_value = stats.shapiro(df_clean[col].dropna())
        print(f'\n{col}:')
        print(f'  Test statistic: {statistic:.4f}')
        print(f'  P-value: {p_value:.4f}')
        print(f'  Distribution: {"Normal" if p_value > 0.05 else "Not Normal"} (at α=0.05)')

## 7. Data Visualization

In [ ]:
# Distribution plots for numerical columns
n_numerical = len(numerical_cols)
if n_numerical > 0:
    fig, axes = plt.subplots(n_numerical, 2, figsize=(15, 5*n_numerical))
    if n_numerical == 1:
        axes = axes.reshape(1, -1)
    
    for idx, col in enumerate(numerical_cols):
        # Histogram
        axes[idx, 0].hist(df_clean[col].dropna(), bins=30, edgecolor='black', alpha=0.7)
        axes[idx, 0].set_title(f'Distribution of {col}')
        axes[idx, 0].set_xlabel(col)
        axes[idx, 0].set_ylabel('Frequency')
        axes[idx, 0].grid(True, alpha=0.3)
        
        # Box plot
        axes[idx, 1].boxplot(df_clean[col].dropna())
        axes[idx, 1].set_title(f'Box Plot of {col}')
        axes[idx, 1].set_ylabel(col)
        axes[idx, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Correlation heatmap
if len(numerical_cols) > 1:
    plt.figure(figsize=(10, 8))
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
                square=True, linewidths=1, cbar_kws={"shrink": 0.8})
    plt.title('Correlation Heatmap', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
# Categorical column visualizations
n_categorical = len(categorical_cols)
if n_categorical > 0:
    fig, axes = plt.subplots(n_categorical, 1, figsize=(12, 5*n_categorical))
    if n_categorical == 1:
        axes = [axes]
    
    for idx, col in enumerate(categorical_cols):
        value_counts = df_clean[col].value_counts()
        axes[idx].bar(range(len(value_counts)), value_counts.values, 
                      tick_label=value_counts.index, edgecolor='black', alpha=0.7)
        axes[idx].set_title(f'Distribution of {col}', fontsize=14, fontweight='bold')
        axes[idx].set_xlabel(col)
        axes[idx].set_ylabel('Count')
        axes[idx].grid(True, alpha=0.3, axis='y')
        
        # Add value labels on bars
        for i, v in enumerate(value_counts.values):
            axes[idx].text(i, v + max(value_counts.values)*0.01, str(v), 
                          ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Pairplot for numerical columns (if not too many)
if len(numerical_cols) > 1 and len(numerical_cols) <= 5:
    print('Creating pairplot...')
    sns.pairplot(df_clean[numerical_cols], diag_kind='kde', plot_kws={'alpha': 0.6})
    plt.suptitle('Pairplot of Numerical Variables', y=1.02, fontsize=16, fontweight='bold')
    plt.show()

In [ ]:
# Time series plot (if date column exists)
date_cols = df_clean.select_dtypes(include=['datetime64']).columns
if len(date_cols) > 0 and len(numerical_cols) > 0:
    date_col = date_cols[0]
    plt.figure(figsize=(15, 6))
    for col in numerical_cols:
        plt.plot(df_clean[date_col], df_clean[col], marker='o', markersize=3, label=col, alpha=0.7)
    plt.title('Time Series Analysis', fontsize=16, fontweight='bold')
    plt.xlabel('Date')
    plt.ylabel('Value')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 8. Conclusions

Summary of key findings from the analysis:

In [ ]:
# Generate summary report
print('='*60)
print('DATASET ANALYSIS SUMMARY')
print('='*60)
print(f'\nDataset Shape: {df_clean.shape[0]} rows × {df_clean.shape[1]} columns')
print(f'\nNumerical Columns: {len(numerical_cols)}')
print(f'Categorical Columns: {len(categorical_cols)}')
print(f'Date/Time Columns: {len(date_cols)}')

if len(numerical_cols) > 0:
    print('\nNumerical Data Summary:')
    for col in numerical_cols:
        print(f'  - {col}: mean={df_clean[col].mean():.2f}, std={df_clean[col].std():.2f}')

if len(categorical_cols) > 0:
    print('\nCategorical Data Summary:')
    for col in categorical_cols:
        print(f'  - {col}: {df_clean[col].nunique()} unique values')

print('\nData Quality:')
print(f'  - Missing values: {df_clean.isnull().sum().sum()}')
print(f'  - Duplicate rows: {df_clean.duplicated().sum()}')

print('\n' + '='*60)
print('Analysis complete!')
print('='*60)

## Next Steps

Based on your analysis, consider:
- Feature engineering for machine learning
- Additional statistical tests
- Advanced visualizations
- Predictive modeling
- Export cleaned data for further use

In [ ]:
# Export cleaned data (optional)
# df_clean.to_csv('cleaned_dataset.csv', index=False)
# print('Cleaned dataset exported successfully!')